<a href="https://colab.research.google.com/github/Wizako-01/Radiomics-Aware-Deep-Learning-Enhancement-of-Simulated-Low-Dose-Lung-CT/blob/main/kaggle_lung_radiomics_colab_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Radiomic-preserving simulated low-dose lung CT enhancement
Research pilot; paired input/target data are created synthetically from each CT.



In [1]:
import os, sys, subprocess, json, random, hashlib, warnings
from pathlib import Path
import numpy as np

def ensure(pkg, module=None):
    import importlib.util
    if importlib.util.find_spec(module or pkg.replace('-', '_')) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

ensure('nibabel')

import nibabel as nib
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.stats import wilcoxon



In [2]:
SEED = 20261025
PATCH = 128
PATCHES_PER_CASE = 16
EPOCHS = 8
BATCH = 16
LEARNING_RATE = 2e-4
MAX_CASES = None              # Set e.g. 20 for a quick smoke test
HU_LOW, HU_HIGH = -1000., 400.
NOISE_SD_HU = 65.           # Synthetic image-domain noise; NOT a dose percentage
FEATURE_LOSS_WEIGHT = 0.2
OUTPUT = Path('/kaggle/working/radiomics_preservation') if Path('/kaggle/working').exists() else Path('./radiomics_preservation')
OUTPUT.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)



Device: cuda


## 1. Discover CT and segmentation pairs automatically
Pairing is performed by patient folder and image/mask attributes, not by
relying on a fixed Kaggle mount name. Stop if a folder is ambiguous.



In [6]:
# ============================================================
# STEP 3: DATASET DISCOVERY AND PATIENT PAIRING
# Medical Decathlon Lung Tumor Segmentation
# ============================================================

from pathlib import Path
import numpy as np
import nibabel as nib
import kagglehub

# 1. Locate the dataset
dataset_path = kagglehub.dataset_download(
    "jadhavrajveer/medical-decathlon-lung-tumor-segmentation"
)

ROOT = Path(dataset_path) / "Lung-Tumor-Segmentation"

assert ROOT.exists(), f"Dataset folder not found: {ROOT}"

print("Dataset location:", ROOT)

# 2. Identify patient folders
patient_folders = sorted(
    [p for p in ROOT.iterdir() if p.is_dir() and p.name.isdigit()],
    key=lambda p: int(p.name)
)

print("Patient folders found:", len(patient_folders))

# 3. Match images and tumour masks
pairs = []

for patient_folder in patient_folders:

    patient_id = patient_folder.name

    image_path = patient_folder / "data" / f"{patient_id}_data.nii"
    mask_path = patient_folder / "label" / f"{patient_id}_mask.nii"

    if not image_path.exists() or not mask_path.exists():
        print(f"Skipping patient {patient_id}: Missing image or mask")
        continue

    # Read image metadata without loading the entire volume
    image = nib.load(str(image_path))
    mask = nib.load(str(mask_path))

    # Check dimensions
    if image.shape != mask.shape:
        print(f"Skipping patient {patient_id}: Shape mismatch")
        continue

    # Check spatial alignment
    if not np.allclose(image.affine, mask.affine, atol=1e-3):
        print(f"Skipping patient {patient_id}: Affine mismatch")
        continue

    pairs.append((patient_id, image_path, mask_path))

# 4. Verify dataset
print("\n" + "=" * 50)
print("DATASET VERIFICATION RESULTS")
print("=" * 50)

print("Total patient folders:", len(patient_folders))
print("Successfully paired patients:", len(pairs))

assert len(pairs) > 0, "No valid CT/mask pairs found!"

# 5. Display first patient information
patient_id, image_path, mask_path = pairs[0]

sample_image = nib.load(str(image_path))
sample_mask = nib.load(str(mask_path))

print("\nFIRST PATIENT")
print("Patient ID:", patient_id)
print("CT dimensions:", sample_image.shape)
print("Mask dimensions:", sample_mask.shape)
print("Voxel spacing:", sample_image.header.get_zooms()[:3])

print("\nCT file:", image_path.name)
print("Mask file:", mask_path.name)

# 6. Confirm readiness
print("\n" + "=" * 50)
print("DATASET LOADING SUCCESSFUL!")
print("Ready for patient-level splitting and preprocessing.")
print("=" * 50)


Dataset location: /root/.cache/kagglehub/datasets/jadhavrajveer/medical-decathlon-lung-tumor-segmentation/versions/1/Lung-Tumor-Segmentation
Patient folders found: 63

DATASET VERIFICATION RESULTS
Total patient folders: 63
Successfully paired patients: 63

FIRST PATIENT
Patient ID: 0
CT dimensions: (256, 256, 559)
Mask dimensions: (256, 256, 559)
Voxel spacing: (np.float32(1.0), np.float32(1.0), np.float32(1.0))

CT file: 0_data.nii
Mask file: 0_mask.nii

DATASET LOADING SUCCESSFUL!
Ready for patient-level splitting and preprocessing.


## 2. Patient-level fixed split



In [4]:
import kagglehub
from pathlib import Path

# Download directly to the current Colab runtime
dataset_path = kagglehub.dataset_download(
    "jadhavrajveer/medical-decathlon-lung-tumor-segmentation"
)

ROOT = Path(dataset_path)

# Verify files
files = list(ROOT.rglob("*.nii"))

print("Dataset location:", ROOT)
print("NIfTI files found:", len(files))

100%|██████████| 1.85G/1.85G [00:21<00:00, 94.4MB/s]

Extracting files...


Dataset location: /root/.cache/kagglehub/datasets/jadhavrajveer/medical-decathlon-lung-tumor-segmentation/versions/1
NIfTI files found: 126


In [7]:
rng = np.random.default_rng(SEED)
order = rng.permutation(len(pairs))
n_train = max(1, int(.7 * len(pairs)))
n_val = max(1, int(.15 * len(pairs)))
splits = {'train': [pairs[i] for i in order[:n_train]],
          'val': [pairs[i] for i in order[n_train:n_train+n_val]],
          'test': [pairs[i] for i in order[n_train+n_val:]]}
assert splits['test'] and splits['val']
pd.DataFrame([{'patient': k, 'split': split, 'image': str(im), 'mask': str(ma)}
              for split, items in splits.items() for k, im, ma in items]).to_csv(OUTPUT/'patient_splits.csv', index=False)
print({k: len(v) for k, v in splits.items()})

def load_case(item):
    k, im, ma = item
    img = nib.load(str(im))
    vol = np.asarray(img.dataobj, dtype=np.float32)
    seg = np.asarray(nib.load(str(ma)).dataobj) > 0
    return vol, seg, img.affine

def normalize(arr):
    return 2 * (np.clip(arr, HU_LOW, HU_HIGH) - HU_LOW) / (HU_HIGH-HU_LOW) - 1

def denormalize(arr):
    return (arr + 1) * (HU_HIGH-HU_LOW) / 2 + HU_LOW

def noisy_image(hu, rng):
    # Heteroscedastic approximation: noise depends on HU, not true dose physics.
    scale = NOISE_SD_HU * np.sqrt(np.clip((hu + 1100) / 1100, .3, 2.5))
    return hu + rng.normal(0, 1, hu.shape).astype(np.float32) * scale

def patch_extract(x, cy, cx, patch=PATCH):
    h, w = x.shape
    y0, x0 = int(cy-patch//2), int(cx-patch//2)
    a = np.pad(x, ((patch, patch), (patch, patch)), mode='edge')
    return a[y0+patch:y0+patch+patch, x0+patch:x0+patch+patch]



{'train': 44, 'val': 9, 'test': 10}


## 3. Fixed training patches; tumour-enriched sampling



In [8]:
class TumourPatches(Dataset):
    def __init__(self, items, seed):
        self.data = []
        gen = np.random.default_rng(seed)
        for case in items:
            hu, mask, _ = load_case(case)
            positive_slices = np.flatnonzero(mask.any(axis=(0, 1)))
            if len(positive_slices) == 0: continue
            for j in range(PATCHES_PER_CASE):
                z = int(gen.choice(positive_slices))
                positions = np.argwhere(mask[:, :, z])
                cy, cx = positions[gen.integers(len(positions))]
                clean = patch_extract(hu[:, :, z], cy, cx)
                roi = patch_extract(mask[:, :, z].astype(np.float32), cy, cx)
                noisy = noisy_image(clean, gen)
                self.data.append((normalize(noisy).astype(np.float32),
                                  normalize(clean).astype(np.float32), roi.astype(np.float32)))
        assert self.data, 'No tumour-containing slices found.'
        print('Patches:', len(self.data))
    def __len__(self): return len(self.data)
    def __getitem__(self, ix):
        return tuple(torch.from_numpy(v[None, ...]) for v in self.data[ix])

train_ds = TumourPatches(splits['train'], SEED)
val_ds = TumourPatches(splits['val'], SEED+1)
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                      generator=torch.Generator().manual_seed(SEED), num_workers=0)
val_dl = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=0)



Patches: 704
Patches: 144


## 4. Train standard denoising U-Net and feature-preserving U-Net
Feature regularization uses differentiable tumour intensity moments and gradients.
It is a surrogate, not the exact PyRadiomics texture feature vector.



In [9]:
class Block(nn.Module):
    def __init__(self, a,b):
        super().__init__()
        self.f = nn.Sequential(nn.Conv2d(a,b,3,padding=1), nn.ReLU(inplace=True),
                               nn.Conv2d(b,b,3,padding=1), nn.ReLU(inplace=True))
    def forward(self,x): return self.f(x)

class SmallUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.a=Block(1,16); self.b=Block(16,32); self.c=Block(32,64)
        self.d=Block(96,32); self.e=Block(48,16)
        self.out=nn.Conv2d(16,1,1)
    def forward(self,x):
        a=self.a(x); b=self.b(F.avg_pool2d(a,2)); c=self.c(F.avg_pool2d(b,2))
        d=self.d(torch.cat([F.interpolate(c,size=b.shape[-2:],mode='bilinear',align_corners=False),b],1))
        e=self.e(torch.cat([F.interpolate(d,size=a.shape[-2:],mode='bilinear',align_corners=False),a],1))
        return torch.clamp(x+self.out(e),-1,1)

def roi_statistics(x, mask):
    m = mask
    n = m.sum((2,3),keepdim=True).clamp_min(1)
    mean = (x*m).sum((2,3),keepdim=True)/n
    var = ((x-mean).square()*m).sum((2,3),keepdim=True)/n
    # Horizontal/vertical differences are crude texture proxies.
    dx = ((x[:,:,:,1:]-x[:,:,:,:-1]).abs()*m[:,:,:,1:]).sum()/m[:,:,:,1:].sum().clamp_min(1)
    dy = ((x[:,:,1:,:]-x[:,:,:-1,:]).abs()*m[:,:,1:,:]).sum()/m[:,:,1:,:].sum().clamp_min(1)
    return mean, var, dx, dy

def feature_loss(pred, target, mask):
    a=roi_statistics(pred,mask); b=roi_statistics(target,mask)
    return sum(F.l1_loss(i,j) for i,j in zip(a,b))

def train_one(name, preserve):
    # Both runs start from the identical weights and see identically generated data.
    torch.manual_seed(SEED+99)
    model=SmallUNet().to(DEVICE)
    opt=torch.optim.Adam(model.parameters(),lr=LEARNING_RATE)
    best=np.inf; history=[]
    for ep in range(EPOCHS):
        model.train(); train_losses=[]
        for inp,tgt,mask in train_dl:
            inp,tgt,mask=(q.to(DEVICE) for q in (inp,tgt,mask))
            opt.zero_grad(set_to_none=True)
            out=model(inp)
            loss=F.mse_loss(out,tgt)
            if preserve: loss=loss+FEATURE_LOSS_WEIGHT*feature_loss(out,tgt,mask)
            loss.backward(); opt.step(); train_losses.append(float(loss.item()))
        model.eval(); val_losses=[]
        with torch.no_grad():
            for inp,tgt,mask in val_dl:
                inp,tgt=(q.to(DEVICE) for q in (inp,tgt))
                val_losses.append(float(F.mse_loss(model(inp),tgt).item()))
        v=float(np.mean(val_losses))
        history.append({'model':name,'epoch':ep+1,'train_loss':float(np.mean(train_losses)),'val_mse':v})
        print(f'{name}: epoch {ep+1}/{EPOCHS}, validation MSE={v:.5f}')
        if v<best:
            best=v;torch.save(model.state_dict(),OUTPUT/f'{name}.pt')
    model.load_state_dict(torch.load(OUTPUT/f'{name}.pt',map_location=DEVICE,weights_only=True))
    return model.eval(), history

standard, h1=train_one('standard',False)
preserved, h2=train_one('feature_preserving',True)
pd.DataFrame(h1+h2).to_csv(OUTPUT/'training_history.csv',index=False)



standard: epoch 1/8, validation MSE=0.00875
standard: epoch 2/8, validation MSE=0.00844
standard: epoch 3/8, validation MSE=0.00761
standard: epoch 4/8, validation MSE=0.00482
standard: epoch 5/8, validation MSE=0.00180
standard: epoch 6/8, validation MSE=0.00048
standard: epoch 7/8, validation MSE=0.00018
standard: epoch 8/8, validation MSE=0.00013
feature_preserving: epoch 1/8, validation MSE=0.00871
feature_preserving: epoch 2/8, validation MSE=0.00854
feature_preserving: epoch 3/8, validation MSE=0.00786
feature_preserving: epoch 4/8, validation MSE=0.00647
feature_preserving: epoch 5/8, validation MSE=0.00467
feature_preserving: epoch 6/8, validation MSE=0.00272
feature_preserving: epoch 7/8, validation MSE=0.00125
feature_preserving: epoch 8/8, validation MSE=0.00055


## 5. Full 3D test-set inference, reproducible noise, PyRadiomics
Train and validation patients are excluded from the evaluation.



In [10]:
def enhance_volume(noisy_hu, model):
    result=np.empty_like(noisy_hu,dtype=np.float32)
    model.eval()
    with torch.inference_mode():
        for z in range(noisy_hu.shape[2]):
            x=torch.from_numpy(normalize(noisy_hu[:,:,z]).astype(np.float32))[None,None].to(DEVICE)
            # padding ensures divisibility by 4
            hh,ww=x.shape[-2:]; ph=(-hh)%4; pw=(-ww)%4
            x=F.pad(x,(0,pw,0,ph),mode='replicate')
            y=model(x)[0,0,:hh,:ww].cpu().numpy()
            result[:,:,z]=denormalize(y)
    return result

def radiomics(hu, seg, affine):
    # Exploratory image features, NOT IBSI-standardized PyRadiomics features.
    vals=np.asarray(hu[seg],dtype=np.float64)
    if len(vals)<20: return {}
    sx,sy,sz=np.linalg.norm(affine[:3,:3],axis=0)
    hist,_=np.histogram(np.clip(vals,-1000,400),bins=64,range=(-1000,400))
    probs=hist[hist>0]/len(vals)
    roi=seg.astype(bool)
    edge_values=[]
    for ax in range(3):
        a=np.diff(hu,axis=ax)
        mask_pair=np.take(roi,range(1,roi.shape[ax]),axis=ax) & np.take(roi,range(roi.shape[ax]-1),axis=ax)
        if np.any(mask_pair): edge_values.append(float(np.mean(np.abs(a[mask_pair]))))
    return {
        'exploratory_mean_hu':float(vals.mean()),
        'exploratory_std_hu':float(vals.std()),
        'exploratory_median_hu':float(np.median(vals)),
        'exploratory_p10_hu':float(np.percentile(vals,10)),
        'exploratory_p90_hu':float(np.percentile(vals,90)),
        'exploratory_entropy_64bins':float(-(probs*np.log2(probs)).sum()),
        'exploratory_mean_abs_neighbor_difference':float(np.mean(edge_values)) if edge_values else 0.,
        'exploratory_tumour_volume_mm3':float(roi.sum()*sx*sy*sz),
    }

records=[]; quality=[]
for ix,case in enumerate(splits['test']):
    key=case[0]; print(f'Test case {ix+1}/{len(splits["test"])}: {key}')
    clean,seg,affine=load_case(case)
    gen=np.random.default_rng(SEED+10000+ix)
    noisy=noisy_image(clean,gen)
    images={'reference':clean,'simulated_low_dose':noisy,
            'standard':enhance_volume(noisy,standard),
            'feature_preserving':enhance_volume(noisy,preserved)}
    for variant,vol in images.items():
        roi=vol[seg]; truth=clean[seg]
        mse=float(np.mean((roi-truth)**2)) if len(roi) else np.nan
        psnr=float(20*np.log10((HU_HIGH-HU_LOW)/np.sqrt(max(mse,1e-12))))
        quality.append({'patient':key,'variant':variant,'roi_mse_hu2':mse,'roi_psnr_db':psnr})
        try:
            feats=radiomics(vol,seg,affine)
            for fname,value in feats.items(): records.append({'patient':key,'variant':variant,'feature':fname,'value':value})
        except Exception as e:
            print('Radiomics failed:',key,variant,str(e))
    del clean,seg,noisy,images

pd.DataFrame(quality).to_csv(OUTPUT/'test_image_quality.csv',index=False)
pd.DataFrame(records).to_csv(OUTPUT/'raw_radiomics.csv',index=False)
assert records, 'No radiomics extracted. Inspect test cases and masks.'



Test case 1/10: 3
Test case 2/10: 45
Test case 3/10: 40
Test case 4/10: 36
Test case 5/10: 5
Test case 6/10: 25
Test case 7/10: 22
Test case 8/10: 37
Test case 9/10: 43
Test case 10/10: 23


## 6. Paired radiomic errors and summary
Absolute relative error is undefined near zero; use absolute differences there.



In [11]:
df=pd.DataFrame(records)
wide=df.pivot_table(index=['patient','feature'],columns='variant',values='value',aggfunc='first')
wide=wide.dropna(subset=['reference','standard','feature_preserving']).copy()
for variant in ['simulated_low_dose','standard','feature_preserving']:
    wide[f'{variant}_abs_error']=(wide[variant]-wide['reference']).abs()
    wide[f'{variant}_relative_error']=wide[f'{variant}_abs_error']/(wide['reference'].abs()+1e-6)
wide.reset_index().to_csv(OUTPUT/'feature_errors_by_patient.csv',index=False)
summary=[]
for feature,grp in wide.reset_index().groupby('feature'):
    a=grp['standard_abs_error'].to_numpy()
    b=grp['feature_preserving_abs_error'].to_numpy()
    if len(a)>1 and np.any(a!=b):
        try: p=float(wilcoxon(a,b).pvalue)
        except ValueError: p=np.nan
    else: p=np.nan
    summary.append({'feature':feature,'n_patients':len(grp),
                    'standard_median_abs_error':float(np.median(a)),
                    'preserving_median_abs_error':float(np.median(b)),
                    'paired_wilcoxon_p_exploratory':p,
                    'improvement_fraction':float(np.mean(b<a))})
pd.DataFrame(summary).to_csv(OUTPUT/'radiomics_summary.csv',index=False)

configuration={'seed':SEED,'patch':PATCH,'patches_per_case':PATCHES_PER_CASE,
               'epochs':EPOCHS,'batch':BATCH,'learning_rate':LEARNING_RATE,
               'HU_window':[HU_LOW,HU_HIGH],'noise_sd_hu':NOISE_SD_HU,
               'feature_loss_weight':FEATURE_LOSS_WEIGHT,
               'simulation_note':'Synthetic heteroscedastic image-domain noise; NOT an acquired or scanner-calibrated low-dose CT.',
               'split_patients':{k:[p[0] for p in v] for k,v in splits.items()},
               'torch':torch.__version__,'numpy':np.__version__,'nibabel':nib.__version__}
(OUTPUT/'configuration.json').write_text(json.dumps(configuration,indent=2))
print('Finished. Results saved to:',OUTPUT)
print(pd.DataFrame(quality).groupby('variant')['roi_psnr_db'].agg(['mean','std','count']))
print(pd.DataFrame(summary).sort_values('improvement_fraction',ascending=False).head(10).to_string(index=False))


Finished. Results saved to: radiomics_preservation
                          mean       std  count
variant                                        
feature_preserving   41.040521  0.295573     10
reference           182.922561  0.000000     10
simulated_low_dose   26.660232  0.069983     10
standard             51.237847  0.199781     10
                                 feature  n_patients  standard_median_abs_error  preserving_median_abs_error  paired_wilcoxon_p_exploratory  improvement_fraction
              exploratory_entropy_64bins          10                   0.442861                     1.300483                       0.001953                   0.0
exploratory_mean_abs_neighbor_difference          10                   4.097427                     7.992287                       0.001953                   0.0
                     exploratory_mean_hu          10                   1.312440                     4.388912                       0.001953                   0.0
             